# Tugas Terstruktur 4

## Perbandingan Model Linier dan Model Berbasis Pohon

**Mata Kuliah** Data Science (TI24425) &middot; 3 sks (Teori)
**Program Studi** Teknologi Informasi &middot; Politeknik Negeri Madiun
**Semester** Genap &middot; Tahun Akademik 2026/2027
**Cakupan materi** Pertemuan 4, 6, 9, dan 10
**Bobot** 4% dari nilai akhir
**Bentuk** Kerja kelompok, tiga orang &middot; diberikan pada pertemuan 10

**Materi rujukan** Pertemuan 4, 6, 9, dan 10 &middot; **Sub-CPMK 9**

---

### Identitas Kelompok

| | Nama Lengkap | NPM | Peran | Bagian |
|---|---|---|---|---|
| 1 | | | Penguji Model Linier | Bagian A |
| 2 | | | Penguji Model Pohon | Bagian B |
| 3 | | | Pembanding Terkendali | Bagian C |

| | |
|---|---|
| **Kelas** | |
| **Judul dataset** | |
| **Tanggal pengumpulan** | |


---

## Petunjuk Pengerjaan

### Kesinambungan dengan tugas sebelumnya

Tugas ini memakai **dataset yang sama** dengan Tugas Terstruktur 1. Jangan berganti dataset. Seluruh rangkaian tugas terstruktur dirancang menumpuk, dan hasilnya menjadi bekal langsung bagi Studi Kasus Akhir pada pertemuan 14 dan 15.

### Cara mengerjakan

- Setiap anggota mengerjakan **satu bagian** sesuai perannya. **Bagian D dikerjakan bersama.**
- Sel bertanda `[KODE]` diisi kode Python. Sel bertanda `[URAIAN]` diisi tulisan Anda sendiri.
- **Kode yang berjalan tanpa uraian tidak memperoleh nilai.** Yang dinilai adalah penalaran.
- Jangan menghapus sel pertanyaan. Tulis jawaban tepat di bawahnya.

### Status kode pada mata kuliah teori

Mata kuliah ini adalah mata kuliah teori. Kegiatan berbasis komputer berlangsung di luar jam tatap muka sebagai **Penugasan Terstruktur**. Kode yang diminta sengaja dibuat sederhana dan dapat dikerjakan dengan menyesuaikan nama kolom. Yang dinilai tetap kualitas analisis.

### Menyimpan sebagai PDF

1. Jalankan seluruh sel: **Run &rarr; Run All Cells**. Pastikan tidak ada pesan galat.
2. **File &rarr; Print** (`Ctrl` + `P`), pilih tujuan **Save as PDF**.
3. Beri nama `TT4_<Kelas>_<NamaKelompok>.pdf`, unggah bersama berkas `.ipynb` ke LMS.

### Penggunaan AI generatif

Diperbolehkan sebagai alat bantu, dengan syarat dicantumkan pada Lampiran di akhir berkas: bagian mana yang dibantu dan bagaimana hasilnya Anda verifikasi.


> **Aturan pembanding yang adil.** Kedua keluarga model wajib dinilai memakai **skema pembagian data dan metrik yang sama**. Membandingkan model dengan prosedur evaluasi berbeda membuat kesimpulannya tidak berarti.

---

## Persiapan

In [ ]:
# [KODE] Persiapan pustaka — jalankan apa adanya
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

pd.set_option('display.max_columns', 50); pd.set_option('display.width', 120)
plt.rcParams['figure.figsize'] = (7.5, 4); plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True; plt.rcParams['grid.alpha'] = 0.3
np.random.seed(42)
print('Pustaka siap.')

In [ ]:
# [KODE] Pemuatan dan penyiapan data — sesuaikan tiga baris pertama saja
NAMA_BERKAS  = 'nama_berkas_dataset.csv'
PEMISAH      = ','
KOLOM_TARGET = 'ganti_dengan_nama_kolom_target'

df = pd.read_csv(NAMA_BERKAS, sep=PEMISAH)

def siapkan(df, target, maks_kategori=15):
    """Menyiapkan X dan y: buang baris tanpa target, encoding kategorik, isi nilai hilang numerik."""
    d = df.dropna(subset=[target]).copy()
    y = d[target]
    X = d.drop(columns=[target])
    # buang kolom kategorik dengan terlalu banyak nilai unik (kemungkinan penanda identitas)
    buang = [c for c in X.columns
             if X[c].dtype == 'object' and X[c].nunique() > maks_kategori]
    if buang:
        print('Kolom dikeluarkan karena terlalu banyak nilai unik:', buang)
        X = X.drop(columns=buang)
    X = pd.get_dummies(X, drop_first=True)              # one-hot untuk kategorik
    X = X.fillna(X.median(numeric_only=True))            # isi nilai hilang numerik
    return X, y

X, y = siapkan(df, KOLOM_TARGET)
print('Ukuran X :', X.shape)
print('Ukuran y :', y.shape)
print('Tipe target:', 'kategorik / klasifikasi' if y.nunique() <= 10 else 'numerik / regresi')

In [ ]:
# [KODE] Menetapkan skema evaluasi bersama — dipakai oleh Bagian A, B, dan C
KLASIFIKASI = y.nunique() <= 10

if KLASIFIKASI:
    y_kerja = pd.factorize(y)[0]
    METRIK  = 'accuracy'
else:
    y_kerja = y.values
    METRIK  = 'neg_mean_squared_error'

X_latih, X_uji, y_latih, y_uji = train_test_split(
    X, y_kerja, test_size=0.20, random_state=42,
    stratify=y_kerja if KLASIFIKASI else None)

LIPATAN = KFold(5, shuffle=True, random_state=42)

print('Jenis masalah  :', 'klasifikasi' if KLASIFIKASI else 'regresi')
print('Metrik bersama :', METRIK)
print('Data latih     :', X_latih.shape, ' ·  Data uji:', X_uji.shape)

---
---

# Bagian A &mdash; Model Linier

**Dikerjakan oleh Anggota 1 &middot; Penguji Model Linier**

In [ ]:
# [KODE] Melatih model linier dengan penskalaan di dalam Pipeline
from sklearn.linear_model import LogisticRegression, Ridge

if KLASIFIKASI:
    pipa_linier = Pipeline([('skala', StandardScaler()),
                            ('model', LogisticRegression(max_iter=3000))])
else:
    pipa_linier = Pipeline([('skala', StandardScaler()),
                            ('model', Ridge(alpha=1.0))])

skor_linier = cross_val_score(pipa_linier, X_latih, y_latih, cv=LIPATAN, scoring=METRIK)
pipa_linier.fit(X_latih, y_latih)

print('Skor cross-validation tiap lipatan :', np.round(skor_linier, 4))
print('Rata-rata  :', round(skor_linier.mean(), 4))
print('Simpangan  :', round(skor_linier.std(), 4))

In [ ]:
# [KODE] Sepuluh koefisien dengan pengaruh terbesar
koef = pipa_linier.named_steps['model'].coef_
koef = koef.ravel()
tabel_koef = (pd.DataFrame({'fitur': X_latih.columns, 'koefisien': np.round(koef, 4)})
                .assign(besaran=lambda d: d['koefisien'].abs())
                .sort_values('besaran', ascending=False)
                .head(10).drop(columns='besaran'))
print(tabel_koef.to_string(index=False))

**[URAIAN A.1]** Jawab kelima pertanyaan berikut.

1. Berapa rata-rata dan simpangan skor cross-validation? Apakah kinerjanya stabil antar-lipatan?
2. Pilih **tiga** koefisien terbesar. Untuk masing-masing, tafsirkan artinya **dalam bahasa masalah Anda**, bukan sekadar menyebut angkanya. Bila modelnya logistic regression, ingat bahwa koefisien menambah log-odds, bukan probabilitas.
3. Adakah koefisien yang **tandanya berlawanan** dengan dugaan Anda? Bila ada, apa kemungkinan penyebabnya?
4. Mengapa penskalaan diletakkan **di dalam** `Pipeline` dan bukan dijalankan lebih dulu pada seluruh data?
5. Sebutkan satu pola pada data Anda yang **tidak mungkin** ditangkap model linier ini. Kaitkan dengan istilah **bias induktif** dari pertemuan 3.

> _Tulis jawaban Anda di sini._


---
---

# Bagian B &mdash; Model Berbasis Pohon

**Dikerjakan oleh Anggota 2 &middot; Penguji Model Pohon**

In [ ]:
# [KODE] Pohon tunggal dan Random Forest, tanpa penskalaan
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

if KLASIFIKASI:
    pohon = DecisionTreeClassifier(max_depth=4, random_state=42)
    hutan = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
else:
    pohon = DecisionTreeRegressor(max_depth=4, random_state=42)
    hutan = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)

skor_pohon = cross_val_score(pohon, X_latih, y_latih, cv=LIPATAN, scoring=METRIK)
skor_hutan = cross_val_score(hutan, X_latih, y_latih, cv=LIPATAN, scoring=METRIK)
pohon.fit(X_latih, y_latih); hutan.fit(X_latih, y_latih)

print('Pohon tunggal  — rerata: {:.4f}  simpangan: {:.4f}'.format(skor_pohon.mean(), skor_pohon.std()))
print('Random Forest  — rerata: {:.4f}  simpangan: {:.4f}'.format(skor_hutan.mean(), skor_hutan.std()))

In [ ]:
# [KODE] Aturan pohon tunggal — dibatasi tiga lapis agar terbaca
print(export_text(pohon, feature_names=list(X_latih.columns), max_depth=3))

In [ ]:
# [KODE] Sepuluh fitur paling sering dipakai memisahkan data
penting = (pd.DataFrame({'fitur': X_latih.columns,
                         'kepentingan': np.round(hutan.feature_importances_, 4)})
             .sort_values('kepentingan', ascending=False).head(10))
fig, ax = plt.subplots()
ax.barh(penting['fitur'][::-1], penting['kepentingan'][::-1])
ax.set_xlabel('Feature importance'); ax.set_title('Sepuluh fitur teratas menurut Random Forest')
plt.tight_layout(); plt.show()
print(penting.to_string(index=False))

**[URAIAN B.1]** Jawab kelima pertanyaan berikut.

1. Bandingkan pohon tunggal dan Random Forest: mana yang rata-ratanya lebih tinggi, dan mana yang simpangannya lebih kecil? Jelaskan **mengapa** ensemble berperilaku demikian, dengan merujuk istilah **variance**.
2. Baca keluaran `export_text`. Tuliskan **satu aturan lengkap** dari akar sampai daun dalam kalimat biasa, seolah menjelaskannya kepada petugas lapangan.
3. Apakah aturan itu masuk akal menurut pemahaman Anda terhadap masalahnya? Bila janggal, apa dugaan penyebabnya?
4. Perhatikan grafik feature importance. Apakah urutannya sejalan dengan koefisien terbesar pada Bagian A? Bila berbeda, apa kemungkinan penyebabnya?
5. **Peringatan penting.** Feature importance yang tinggi **bukan** bukti hubungan sebab-akibat. Jelaskan apa sebenarnya yang diukur oleh angka tersebut.

> _Tulis jawaban Anda di sini._


---
---

# Bagian C &mdash; Perbandingan Terkendali

**Dikerjakan oleh Anggota 3 &middot; Pembanding Terkendali**

In [ ]:
# [KODE] Perbandingan seluruh model dengan skema evaluasi yang sama
model_uji = {'Linier': pipa_linier, 'Pohon tunggal': pohon, 'Random Forest': hutan}
baris = []
for nama, m in model_uji.items():
    s = cross_val_score(m, X_latih, y_latih, cv=LIPATAN, scoring=METRIK)
    baris.append({'model': nama,
                  'rerata_cv': round(s.mean(), 4),
                  'simpangan_cv': round(s.std(), 4),
                  'terburuk': round(s.min(), 4),
                  'terbaik': round(s.max(), 4)})
tabel = pd.DataFrame(baris).sort_values('rerata_cv', ascending=False)
print(tabel.to_string(index=False))

In [ ]:
# [KODE] Uji sekali pada data uji — dijalankan hanya SATU KALI, di akhir
print('Skor pada data uji (dibuka satu kali):')
for nama, m in model_uji.items():
    print(f'  {nama:15s}: {m.score(X_uji, y_uji):.4f}')
print()
print('Bandingkan dengan rerata cross-validation di atas.')
print('Selisih besar antara keduanya adalah tanda yang harus Anda jelaskan.')

**[URAIAN C.1]** Jawab keempat pertanyaan berikut.

1. Model mana yang **rata-rata cross-validation**-nya tertinggi? Berapa selisihnya dengan model kedua?
2. Apakah selisih itu **lebih besar** daripada simpangan antar-lipatan? Bila tidak, apa artinya bagi kesimpulan Anda?
3. Bandingkan skor cross-validation dengan skor pada data uji. Adakah selisih yang mencolok? Apa dugaan penyebabnya?
4. Mengapa data uji hanya boleh dibuka **satu kali**, dan apa yang rusak seandainya Anda kembali menyetel model setelah melihat angka itu?

> _Tulis jawaban Anda di sini._


## C.2 Tabel Perbandingan Karakteristik

**[URAIAN C.2]** Isi tabel berikut berdasarkan pengalaman Anda mengerjakan Bagian A dan B pada dataset ini — **bukan** menyalin dari slide.

| Aspek | Model linier | Model berbasis pohon | Bukti dari pekerjaan kelompok kami |
|---|---|---|---|
| Perlukah penskalaan? | | | |
| Perlukah encoding kategorik? | | | |
| Kemampuan menangkap hubungan tidak lurus | | | |
| Keterbacaan hasil | | | |
| Kemampuan memprediksi di luar rentang data latih | | | |
| Kestabilan antar-lipatan | | | |


---
---

# Bagian D &mdash; Justifikasi Pemilihan

**Dikerjakan bersama oleh ketiga anggota.**

## D.1 Empat Pertanyaan Sebelum Memutuskan

**[URAIAN D.1]** Jawab keempat pertanyaan panduan dari pertemuan 10 untuk konteks kelompok Anda.

| Pertanyaan | Jawaban kelompok kami | Akibatnya bagi pilihan model |
|---|---|---|
| Apakah hasilnya harus dapat dijelaskan kepada orang awam? | | |
| Apakah datanya berbentuk tabel dengan banyak variabel kategorik? | | |
| Apakah hubungan antar-variabel diduga lurus dan sederhana? | | |
| Apakah perlu memprediksi di luar rentang data yang ada? | | |


## D.2 Keputusan dan Harganya

**[URAIAN D.2]** Susun keputusan akhir kelompok Anda.

| Butir | Isian |
|---|---|
| Model yang kelompok kami pilih | |
| Alasan yang paling menentukan | |
| Berapa poin kinerja yang kami korbankan dibanding model terbaik secara angka? | |
| Apa yang kami peroleh sebagai gantinya? | |
| Kepada siapa keputusan ini harus dapat dipertanggungjawabkan? | |

Kemudian jawab: seandainya model dengan angka terbaik **bukan** yang Anda pilih, bagaimana Anda menjelaskan keputusan itu kepada atasan yang hanya melihat angka?

> _Tulis jawaban Anda di sini._


## D.3 Batas yang Kami Akui

**[URAIAN D.3]** Jawab jujur ketiga pertanyaan berikut.

1. Apakah salah satu model Anda menunjukkan tanda **overfitting**? Sebutkan buktinya dari selisih skor latih, cross-validation, dan uji.
2. Adakah fitur yang menurut Anda **seharusnya diperiksa ulang** karena kepentingannya mencurigakan tinggi? Kaitkan dengan pembahasan kebocoran data pertemuan 9.
3. Apa satu hal yang paling membuat Anda ragu terhadap kesimpulan tugas ini?

> _Tulis jawaban Anda di sini._


## D.4 Pembagian Kerja

| Anggota | Bagian | Perkiraan waktu | Kesulitan terbesar |
|---|---|---|---|
| 1 | Bagian A | | |
| 2 | Bagian B | | |
| 3 | Bagian C | | |
| Bersama | Bagian D | | |


---

# Lampiran &mdash; Pernyataan Penggunaan AI Generatif

| Bagian yang dibantu | Nama alat | Bentuk bantuan | Cara kami memverifikasi |
|---|---|---|---|
| | | | |
| | | | |

Dengan ini kami menyatakan bahwa seluruh analisis, justifikasi, dan kesimpulan dalam berkas ini merupakan hasil penalaran kelompok kami sendiri, dan seluruh bantuan alat AI generatif telah kami cantumkan secara jujur.

| Anggota 1 | Anggota 2 | Anggota 3 |
|---|---|---|
| ( ......................... ) | ( ......................... ) | ( ......................... ) |


---

# Rubrik Penilaian

Total 100 poin, dikonversi menjadi bobot 4% pada nilai akhir.

| Bagian | Kriteria | Poin |
|---|---|---|
| A | Ketepatan menafsirkan koefisien dalam bahasa masalah | 13 |
| A | Pemahaman kewajiban penskalaan dan letak Pipeline | 7 |
| A | Ketepatan menyebut keterbatasan model linier dan kaitannya dengan bias induktif | 10 |
| B | Ketepatan menjelaskan perbedaan pohon tunggal dan ensemble melalui istilah variance | 12 |
| B | Ketepatan membaca satu aturan pohon secara utuh dan menilai kewajarannya | 10 |
| B | Ketepatan menjelaskan bahwa feature importance bukan bukti sebab-akibat | 8 |
| C | Kesahihan perbandingan: skema evaluasi dan metrik yang sama | 10 |
| C | Ketepatan menilai apakah selisih antar-model bermakna dibanding simpangannya | 10 |
| D | Kekuatan justifikasi pemilihan model dengan mengaitkan ciri data dan kebutuhan pengguna | 12 |
| D | Kejujuran mengakui overfitting, kecurigaan kebocoran, dan keraguan | 8 |
| | **Jumlah** | **100** |

### Ketentuan penilaian

- **Kode yang berjalan tanpa uraian bernilai nol** untuk butir yang bersangkutan.
- Jawaban umum yang dapat dipakai untuk dataset mana pun **tidak memperoleh nilai penuh**.
- Menyebut suatu hal keliru **tanpa menjelaskan mengapa** hanya memperoleh separuh poin.
- Mengakui keterbatasan secara jujur **menambah** nilai; menutupinya mengurangi nilai.
- Keterlambatan dikenakan pengurangan 10% nilai per hari kerja, maksimal tiga hari kerja.

### Pembobotan khusus tugas ini

Sesuai kriteria yang disampaikan pada pertemuan 10: **ketepatan menjelaskan mekanisme kedua keluarga model 30%**, **kesesuaian analisis dengan karakteristik data 30%**, dan **kekuatan justifikasi pemilihan 40%**. Menyebut satu model “lebih baik” tanpa mengaitkannya pada ciri data dan kebutuhan pengguna tidak memperoleh nilai penuh.